# Module 2 Simple Code Along

Goal: understand why `dict` and JSON matter so much in Python.

Core idea:

> Classes are how our code thinks. Dicts and JSON are how systems talk.

## 1. A product as a Python dict

A `dict` is Python's basic shape for structured data.

In [1]:
product = {
    "id": 1,
    "name": "Keyboard",
    "category": "Electronics",
    "price": 5499.0,
    "in_stock": True,
    "tags": ["keyboard", "mechanical"],
}

print(product["name"])
print(product["price"])

Keyboard
5499.0


## 2. List vs dict lookup

A list is good for "all products". A dict is good for "give me product 2 now".

In [2]:
products = [
    {"id": 1, "name": "Cable", "price": 499.0},
    {"id": 2, "name": "Keyboard", "price": 5499.0},
    {"id": 3, "name": "Mouse", "price": 999.0},
]

# List lookup: scan each product
for item in products:
    if item["id"] == 2:
        print("found in list:", item["name"])

found in list: Keyboard


In [3]:
# Dict lookup: build an index by id
products_by_id = {
    item["id"]: item
    for item in products
}

print(products_by_id[2]["name"])

Keyboard


## 3. A class for behavior

A dict is flexible data. A class gives the data rules and operations.

In [4]:
class Product:
    def __init__(self, id, name, category, price):
        if price < 0:
            raise ValueError("price must be non-negative")
        self.id = id
        self.name = name
        self.category = category
        self.price = price

    def to_dict(self):
        return {
            "id": self.id,
            "name": self.name,
            "category": self.category,
            "price": self.price,
        }


keyboard = Product(2, "Keyboard", "Electronics", 5499.0)
print(keyboard.name)
print(keyboard.to_dict())

Keyboard
{'id': 2, 'name': 'Keyboard', 'category': 'Electronics', 'price': 5499.0}


## 4. JSON is text for systems

JSON looks like a Python dict, but it is text. APIs, files, and LLM tools commonly use JSON-shaped data.

In [5]:
import json

row = keyboard.to_dict()

json_text = json.dumps(row, indent=2)
print(json_text)
print(type(json_text))

{
  "id": 2,
  "name": "Keyboard",
  "category": "Electronics",
  "price": 5499.0
}
<class 'str'>


In [ ]:
back_to_dict = json.loads(json_text)

print(back_to_dict)
print(type(back_to_dict))

Remember:

- `json.dumps(...)` -> Python object to JSON string
- `json.loads(...)` -> JSON string to Python object

## 5. Reading and writing JSON files

Use `pathlib` plus `json` for simple JSON files.

In [6]:
from pathlib import Path

path = Path("products-simple.json")

Path("products-simple.json").write_text(
    json.dumps(products, indent=2)
)

print(path.read_text())

[
  {
    "id": 1,
    "name": "Cable",
    "price": 499.0
  },
  {
    "id": 2,
    "name": "Keyboard",
    "price": 5499.0
  },
  {
    "id": 3,
    "name": "Mouse",
    "price": 999.0
  }
]


In [7]:
loaded_products = json.loads(path.read_text())

print(loaded_products)
print(loaded_products[0]["name"])

[{'id': 1, 'name': 'Cable', 'price': 499.0}, {'id': 2, 'name': 'Keyboard', 'price': 5499.0}, {'id': 3, 'name': 'Mouse', 'price': 999.0}]
Cable


## 6. CSV also becomes dicts

`csv.DictReader` gives one dict per row. CSV values come in as strings.

In [8]:
import csv
import io

csv_text = """id,name,price
1,Cable,499
2,Keyboard,5499
3,Mouse,999
"""

reader = csv.DictReader(io.StringIO(csv_text))

for row in reader:
    print(row, type(row["price"]))

{'id': '1', 'name': 'Cable', 'price': '499'} <class 'str'>
{'id': '2', 'name': 'Keyboard', 'price': '5499'} <class 'str'>
{'id': '3', 'name': 'Mouse', 'price': '999'} <class 'str'>


In [9]:
# Convert CSV strings into useful Python types
reader = csv.DictReader(io.StringIO(csv_text))

clean_rows = []
for row in reader:
    clean_rows.append({
        "id": int(row["id"]),
        "name": row["name"],
        "price": float(row["price"]),
    })

print(clean_rows)

[{'id': 1, 'name': 'Cable', 'price': 499.0}, {'id': 2, 'name': 'Keyboard', 'price': 5499.0}, {'id': 3, 'name': 'Mouse', 'price': 999.0}]


## 7. Mini recap

Use this mental model:

```text
JSON file/API/LLM output
    -> json.loads or json.load
    -> Python dict/list
    -> class or Pydantic model for rules
    -> dict/list
    -> json.dumps or json.dump
```

Short version:

- `dict` is Python's flexible structured data shape.
- JSON is the outside world's structured data shape.
- Classes hold behavior and rules.
- Pydantic later gives class-like objects plus validation.